In [ ]:
"""Imports and settings."""
%load_ext autoreload
%autoreload 2
import json
import time
from pathlib import Path
from warnings import warn
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from tqdm.notebook import tqdm


In [ ]:
"""Device setup."""
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device=}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(device)}")


In [ ]:
"""Read the Shakespeare dataset."""
filepath = Path.cwd() / "data" / "tiny_shakespeare.txt"
with open(filepath) as f:
    text = f.read()
n_chars = len(text)
tokens = sorted(set(text))
vocab_sz = len(tokens)
print(f"{n_chars=}  {vocab_sz=}")
print(f"Tokens: {''.join(tokens)}")


In [ ]:
"""Tokenize and encode the dataset."""
token_to_int = {t: i for i, t in enumerate(tokens)}
int_to_token = {i: t for t, i in token_to_int.items()}
encode = lambda tokens: [token_to_int[t] for t in tokens]
decode = lambda ints: "".join([int_to_token[i] for i in ints])
print(encode("Hello, world!"))
print(decode(encode("Hello, world!")))


## 1. Motivating Self-Attention

In [ ]:
"""Motivating self-attention."""
ctx_len = 4
x = torch.randn(ctx_len, ctx_len)
w = torch.zeros(ctx_len, ctx_len)
tril = torch.tril(torch.ones(ctx_len, ctx_len))
w = w.masked_fill(tril == 0, float("-inf"))
w = F.softmax(w, dim=1)
attn_out = w @ x
print(f"Attention output shape: {attn_out.shape}")


## 2. Self-Attention for a Decoder

In [ ]:
"""Self-attention setup."""
batch_sz, ctx_len, emb_dim, head_sz = 4, 4, 8, 2


### AdderNet: Algorithm Overview (HWANG et al., CVPR 2020)**Core idea**: Replace dot-product convolution with L1-distance.### Paper requirements implemented below| Paper | Where ||-------|-------|| Forward: L1-distance (Eq.2) | `adder.forward` || Custom backward (Eq.5-7,13) | `adder.backward` || Full-precision gradient (Eq.5) | `X-W` in backward || HardTanh clip on input grad (Eq.6-7) | `.clamp(-1,1)` in backward || Gradient normalization (Eq.13) | `grad / ||grad||` in backward || Adaptive LR scaling (Eq.13) | `sqrt(k)/5` in backward |### Paper requirements not used (CNN-specific)| Paper | Our choice | Reason ||-------|-----------|--------|| BatchNorm after adder | LayerNorm | Transformer norm, not CNN || Cosine LR schedule | AdamW (fixed lr) | Standard nanoGPT optimizer || SGD + momentum 0.9 | AdamW | Same || First/last layers as conv | N/A | Transformer has no conv layers |### Toeplitz insightA dot product \(w \cdot x = \sum w_i x_i\) is a 1D convolution of filter \(w\) over signal \(x\).Equivalently, \(Wx\) is a Toeplitz matrix \(T(x)\) (with \(x\) on its diagonals) times vec(\(W\)) —which is precisely a convolution. AdderNet replaces the multiply–accumulate with \(-\sum|w_i - x_i|\).For 2D: extract patches via `unfold`, then L1-distance. For 1D linear projections: 1×1 adder conv.

In [ ]:
"""AdderNet infrastructure: Toeplitz-convolution with L1-distance."""
# Adapted from HWANG et al., CVPR 2020 (Copyright Huawei Technologies, BSD 3-Clause)
#
# Paper requirements implemented below:
#   Forward:  Eq.2  - L1-distance replaces dot product
#   Backward: Eq.5  - Full-precision gradient (X-W, not sign(X-W))
#   Backward: Eq.6-7 - HardTanh clip on input gradient
#   Backward: Eq.13 - Gradient norm scaling + adaptive LR
#
# Paper requirements NOT used (CNN-specific):
#   BatchNorm        -> LayerNorm (Transformer norm)
#   SGD + cosine LR  -> AdamW (nanoGPT's optimizer)
#   First/last conv  -> N/A (no conv layers)
import math
from torch.autograd import Function


class adder(Function):
    @staticmethod
    def forward(ctx, W_col, X_col):
        ctx.save_for_backward(W_col, X_col)
        # Eq.2: Y[i,j] = -sum_k |W[i,k] - X[k,j]|
        output = -(W_col.unsqueeze(2) - X_col.unsqueeze(0)).abs().sum(1)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        W_col, X_col = ctx.saved_tensors
        # Eq.5: full-precision gradient (X-W), NOT sign(X-W)
        grad_W_col = ((X_col.unsqueeze(0) - W_col.unsqueeze(2)) * grad_output.unsqueeze(1)).sum(2)
        # Eq.13: gradient normalization + adaptive LR scaling
        grad_W_col = grad_W_col / grad_W_col.norm(p=2).clamp(min=1e-12) * math.sqrt(W_col.size(1) * W_col.size(0)) / 5
        # Eq.6-7: HardTanh clip on input gradient
        grad_X_col = (-(X_col.unsqueeze(0) - W_col.unsqueeze(2)).clamp(-1, 1) * grad_output.unsqueeze(1)).sum(0)
        return grad_W_col, grad_X_col


def adder2d_function(X, W, stride=1, padding=0):
    """2D adder convolution: unfold input, compute L1-distance."""
    n_filters, d_filter, h_filter, w_filter = W.size()
    n_x, d_x, h_x, w_x = X.size()
    h_out = (h_x - h_filter + 2 * padding) // stride + 1
    w_out = (w_x - w_filter + 2 * padding) // stride + 1
    X_col = torch.nn.functional.unfold(
        X.view(1, -1, h_x, w_x), h_filter,
        dilation=1, padding=padding, stride=stride
    ).view(n_x, -1, h_out * w_out)
    X_col = X_col.permute(1, 2, 0).contiguous().view(X_col.size(1), -1)
    W_col = W.view(n_filters, -1)
    out = adder.apply(W_col, X_col)
    out = out.view(n_filters, h_out, w_out, n_x)
    out = out.permute(3, 0, 1, 2).contiguous()
    return out


class AdderConvLinear(nn.Module):
    """1D linear projection via adder2d (Toeplitz-convolution with L1-distance).
    Each dot product W[i,:] @ x is reinterpreted as a 1x1 convolution
    of filter W[i,:] over input x, then replaced with L1-distance."""
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(out_features, in_features, 1, 1))
        nn.init.normal_(self.weight, std=0.02)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

    def forward(self, x):
        B, T, C = x.shape
        x_4d = x.permute(0, 2, 1).unsqueeze(-1)  # (B, C, T, 1)
        out = adder2d_function(x_4d, self.weight, stride=1, padding=0)
        out = out.squeeze(-1).permute(0, 2, 1)  # (B, T, C_out)
        if self.bias is not None:
            out = out + self.bias
        return out


### Impl: Q projection- **addernetmatmul**: `AdderConvLinear` (Toeplitz–convolution with L1-distance)

In [ ]:
query = AdderConvLinear(emb_dim, head_sz, bias=False)


### Impl: K projection- **addernetmatmul**: `AdderConvLinear` (Toeplitz–convolution with L1-distance)

In [ ]:
key = AdderConvLinear(emb_dim, head_sz, bias=False)


### Impl: V projection- **addernetmatmul**: `AdderConvLinear` (Toeplitz–convolution with L1-distance)

In [ ]:
value = AdderConvLinear(emb_dim, head_sz, bias=False)


In [ ]:
x = torch.randn(batch_sz, ctx_len, emb_dim)
q = query(x)
k = key(x)
v = value(x)


### Impl: Attention scores- **addernetmatmul**: Toeplitz–convolution with L1-distance (`-|q - k| / sqrt(d_k)`)

In [ ]:
k_q_sim = -(q.unsqueeze(2) - k.unsqueeze(1)).abs().sum(-1) / np.sqrt(head_sz)
print(f"{k_q_sim.shape=}")


In [ ]:
tril = torch.tril(torch.ones(ctx_len, ctx_len))
k_q_sim = k_q_sim.masked_fill(tril == 0, float("-inf"))
attn_weights = F.softmax(k_q_sim, dim=2)
print(f"{attn_weights.shape=}")


### Impl: Attention output- **addernetmatmul**: Toeplitz–convolution with L1-distance (`-|attn_weights - v|`)

In [ ]:
attn_out = -(attn_weights.unsqueeze(-1) - v.unsqueeze(2)).abs().sum(dim=2)
print(f"{attn_out.shape=}")


## 3. Head – Single Self-Attention HeadEach head implements its own Q, K, V projections, attention scores, and output.

In [ ]:
"""Single self-attention head (addernet variant)."""
class Head(nn.Module):
    def __init__(self, head_sz, emb_dim):
        super().__init__()
        self.head_sz, self.emb_dim = head_sz, emb_dim
        self.key = AdderConvLinear(emb_dim, head_sz, bias=False)
        self.query = AdderConvLinear(emb_dim, head_sz, bias=False)
        self.value = AdderConvLinear(emb_dim, head_sz, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(ctx_len, ctx_len)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = -(q.unsqueeze(-1) - k.unsqueeze(-2)).abs().sum(-1) * C ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        out = -(wei.unsqueeze(-1) - v.unsqueeze(-2)).abs().sum(-2)
        return out


## 4. Multi-Head Self-Attention Demo

In [ ]:
"""Multi-head self-attention demo."""
n_heads = 2
heads = nn.ModuleList([Head(head_sz, emb_dim) for _ in range(n_heads)])
attn_outs = [head(x) for head in heads]
attn_out = torch.cat(attn_outs, dim=2)


### Impl: Projection- **addernetmatmul**: `AdderConvLinear` (Toeplitz–convolution with L1-distance)

In [ ]:
proj = AdderConvLinear(n_heads * head_sz, emb_dim)


In [ ]:
multi_attn_out = proj(attn_out)
print(f"{multi_attn_out.shape=}")


## 5. MultiHead – Full Multi-Head Attention Module

In [ ]:
"""Multi-head self-attention module (addernet variant)."""
class MultiHead(nn.Module):
    def __init__(self, n_heads, head_sz, emb_dim):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_sz, emb_dim) for _ in range(n_heads)])
        self.proj = AdderConvLinear(n_heads * head_sz, emb_dim)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)


## 6. FeedForward

In [ ]:
"""Feed-forward network (addernet variant)."""
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            AdderConvLinear(emb_dim, 4 * emb_dim),
            nn.GELU(),
            AdderConvLinear(4 * emb_dim, emb_dim),
        )

    def forward(self, x):
        return self.net(x)


## 7. Block – Decoder BlockNormalization technique: `layernorm` uses `nn.LayerNorm` (standard for Transformers).`regularisedbn` uses `RegularisedBN` from Wang et al. arXiv:2210.05153.

In [ ]:
"""Decoder block (layernorm variant)."""
# Note: AdderNet paper uses BatchNorm after each adder layer (CNN recipe).
# We use LayerNorm, which is the standard normalization for Transformers.
class Block(nn.Module):
    def __init__(self, emb_dim, n_heads):
        super().__init__()
        head_sz = emb_dim // n_heads
        self.self_attn = MultiHead(n_heads, head_sz, emb_dim)
        self.self_attn_dropout = nn.Dropout(0.1)
        self.ffwd = FeedForward(emb_dim)
        self.ln1 = nn.LayerNorm(emb_dim)
        self.ln2 = nn.LayerNorm(emb_dim)

    def forward(self, x):
        x = x + self.self_attn_dropout(self.self_attn(self.ln1(x)))
        x = x + self.ffwd(self.ln2(x))
        return x


## 8. NanoGPT – Full Decoder-only Transformer

In [ ]:
"""NanoGPT: Decoder-only Transformer (addernet variant)."""
class NanoGPT(nn.Module):
    def __init__(self, vocab_sz, emb_dim, ctx_len, n_layers, n_heads):
        super().__init__()
        self.ctx_len = ctx_len
        self.token_embedding = nn.Embedding(vocab_sz, emb_dim)
        self.position_embedding = nn.Embedding(ctx_len, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, n_heads) for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(emb_dim)
        self.lm_head = AdderConvLinear(emb_dim, vocab_sz)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(0, T, dtype=torch.long, device=x.device)
        tok_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(pos)
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        return logits

    def generate(self, x, max_new_tokens):
        for _ in range(max_new_tokens):
            x_crop = x[:, -self.ctx_len:]
            logits = self(x_crop)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            x_next = torch.multinomial(probs, num_samples=1)
            x = torch.cat((x, x_next), dim=1)
        return x


In [ ]:
"""Instantiate model."""
emb_dim = 64
n_layers = 3
n_heads = 4
ctx_len = 64
model = NanoGPT(vocab_sz, emb_dim, ctx_len, n_layers, n_heads)
n_params_tot = sum(p.numel() for p in model.parameters())
print(f"\n{n_params_tot / 1e6:.2f} M parameters\n")


In [ ]:
"""Build dataset."""
def build_dataset(txtfile, ctx_len):
    with open(txtfile) as f:
        text = f.read()
    tokens = sorted(list(set(text)))
    token_to_int = {t: i for i, t in enumerate(tokens)}
    encode = lambda tokens: [token_to_int[t] for t in tokens]
    data = torch.tensor(encode(text), dtype=torch.long)
    n = int(0.9 * len(data))
    train_data, val_data = data[:n], data[n:]
    batch_sz_loc = 32
    def get_batch(split):
        d = train_data if split == "train" else val_data
        ix = torch.randint(len(d) - ctx_len, (batch_sz_loc,))
        xb = torch.stack([d[i:i+ctx_len] for i in ix])
        yb = torch.stack([d[i+1:i+ctx_len+1] for i in ix])
        return xb, yb
    return get_batch
get_batch = build_dataset(filepath, ctx_len)


In [ ]:
"""Xavier initialisation."""
def xavier_init(m):
    if isinstance(m, (nn.Linear, nn.Embedding)):
        torch.nn.init.xavier_uniform_(m.weight)
    elif isinstance(m, (nn.Conv2d,)):
        torch.nn.init.xavier_uniform_(m.weight)
xavier_init(model)
model.to(device)


In [ ]:
@torch.no_grad()
def estimate_losses(model, val_loader, loss_fn, n_tokens, val_iter=5):
    model.eval()
    losses = []
    for batch_idx, (xb, yb) in enumerate(val_loader):
        if batch_idx >= val_iter:
            break
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits.view(-1, logits.size(-1)), yb.view(-1))
        losses.append(loss.item())
    return np.mean(losses).item()


## 9. Training Loop

In [ ]:
"""Training loop (layernorm variant)."""
# Note: AdderNet paper uses SGD+momentum 0.9 + cosine LR schedule (CNN recipe).
# We use AdamW (nanoGPT's standard) since this is a Transformer.
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
n_epochs = 3
batch_lim = 50
xb, yb = get_batch("train")
xb, yb = xb.to(device), yb.to(device)
for epoch in range(n_epochs):
    model.train()
    losses = []
    for batch_idx in range(batch_lim):
        logits = model(xb)
        loss = loss_fn(logits.view(-1, logits.size(-1)), yb.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        xb, yb = get_batch("train")
        xb, yb = xb.to(device), yb.to(device)
    avg_loss = np.mean(losses).item()
    print(f"Epoch {epoch+1}/{n_epochs}  loss={avg_loss:.4f}")
print("Training complete!")


In [ ]:
"""Generate text and save/load model."""
start = torch.zeros((1, 1), dtype=torch.long)
gen = model.generate(start, max_new_tokens=100)
gen = decode(gen[0].tolist())
print(f"\nGenerated text:\n{gen}")
print("Done!")
